# Domain F: Recurrent Neural Network Modeling

This notebook addresses two questions:
- **F1:** Can a task-trained RNN (Dale's law constrained, E/I ratio matching data) reproduce cell-type-specific tuning and connectivity?
- **F2:** Can a data-driven RNN trained to predict population ΔF/F learn biologically meaningful representations?

**Dependencies:** `torch` (PyTorch). Install via `pip install torch`.

**Note:** Both models use the actual stimulus parameters from the dataset but are trained on synthetic or simplified tasks. Sections requiring additional data or long training are marked.

In [ ]:
import sys
sys.path.insert(0, '.')

import numpy as np
import pandas as pd
import zarr
import matplotlib.pyplot as plt
import seaborn as sns
from types import SimpleNamespace
from scipy.stats import pearsonr, spearmanr
import warnings
warnings.filterwarnings('ignore')

try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    TORCH_AVAILABLE = True
    print(f"PyTorch version: {torch.__version__}")
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {device}")
except ImportError:
    TORCH_AVAILABLE = False
    print("⚠️ PyTorch not installed. Install via: pip install torch")

from functions.data_loading import load_mouse_zarr, load_zarr_10hz
from functions.analysis import linear_CKA
from functions.models import DalesRNN, PredictiveRNN, TemporalRNN

sns.set_context('talk')
sns.set_style('whitegrid')

# ══════════════════════════════════════════════════════════════════════
# Load data from zarr multimodal stores
# ══════════════════════════════════════════════════════════════════════
ZARR_DIR = 'multimodal_data'
MOUSE_IDS = ['778174', '786297', '797371']
SESSIONS = ['session_1', 'session_2', 'session_3']

adata_list = [load_mouse_zarr(mid, zarr_dir=ZARR_DIR) for mid in MOUSE_IDS]

obs = pd.concat([a.obs for a in adata_list], ignore_index=True)
X = np.vstack([a.X for a in adata_list])
var = adata_list[0].var.copy()

SUBCLASS_ORDER = ['007 L2/3 IT CTX Glut', '006 L4/5 IT CTX Glut', '022 L5 ET CTX Glut',
                  '052 Pvalb Gaba', '053 Sst Gaba', '046 Vip Gaba', '049 Lamp5 Gaba']
SUBCLASS_COLORS = {'007 L2/3 IT CTX Glut': '#1f77b4', '006 L4/5 IT CTX Glut': '#2ca02c',
                   '022 L5 ET CTX Glut': '#9467bd', '052 Pvalb Gaba': '#d62728',
                   '053 Sst Gaba': '#ff7f0e', '046 Vip Gaba': '#e377c2', '049 Lamp5 Gaba': '#8c564b'}
SUBCLASS_SHORT = {'007 L2/3 IT CTX Glut': 'L2/3 IT', '006 L4/5 IT CTX Glut': 'L4/5 IT',
                  '022 L5 ET CTX Glut': 'L5 ET', '052 Pvalb Gaba': 'Pvalb',
                  '053 Sst Gaba': 'Sst', '046 Vip Gaba': 'Vip', '049 Lamp5 Gaba': 'Lamp5'}
present_subclasses = [s for s in SUBCLASS_ORDER if s in obs['subclass_name'].unique()]

# Backward-compatible adata object for downstream cells
adata = SimpleNamespace(X=X, obs=obs, var=var, n_obs=len(obs), n_vars=X.shape[1])

orientations = np.array([0, 45, 90, 135, 180, 225, 270, 315])
contrasts = np.array([0.05, 0.1, 0.2, 0.4, 0.8])
tfs = np.array([1, 2, 4, 8, 15])

# ── Data-derived E/I ratio ──
sc_counts = obs['subclass_name'].value_counts()
exc_count = sum(sc_counts.get(s, 0) for s in present_subclasses if 'Glut' in s)
inh_count = sum(sc_counts.get(s, 0) for s in present_subclasses if 'Gaba' in s)
print(f"\nExcitatory: {exc_count} ({exc_count/len(obs):.1%}), Inhibitory: {inh_count} ({inh_count/len(obs):.1%})")
print(f"E/I ratio: {exc_count/max(inh_count,1):.1f}:1")

## F2: Data-Driven RNN — Predict Population ΔF/F

Train an RNN to predict the actual population neural activity from stimulus inputs and running speed. Compare learned internal representations to real population coding geometry using CKA and RSA.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# F2.1  Data-driven RNN: predict population ΔF/F from stimulus
# ══════════════════════════════════════════════════════════════════════

if TORCH_AVAILABLE:
    # ── Prepare data: use single mouse, subsample cells for tractability ──
    mouse_pick = obs['mouse_id'].unique()[np.argmax(
        [np.sum(obs['mouse_id'].values == m) for m in obs['mouse_id'].unique()])]
    m_mask = obs['mouse_id'].values == mouse_pick
    m_X = X[m_mask]
    n_cells_mouse = m_mask.sum()
    
    # Subsample to 100 cells if needed
    n_target = min(100, n_cells_mouse)
    np.random.seed(42)
    cell_idx = np.sort(np.random.choice(n_cells_mouse, n_target, replace=False))
    target_dff = m_X[cell_idx]  # (n_target, n_trials)
    target_subclasses = obs['subclass_name'].values[m_mask][cell_idx]
    
    # Stimulus input features per trial: cos(ori), sin(ori), contrast, log(TF), running
    ori_rad = np.deg2rad(var['orientation'].values.astype(float))
    running_speed = var['avg_running'].values.astype(float)
    running_speed = np.clip(running_speed / 30, -1, 2)  # normalize
    
    stim_inputs = np.column_stack([
        np.cos(ori_rad),
        np.sin(ori_rad),
        var['contrast'].values.astype(float) / 0.8,
        np.log2(var['temporal_frequency'].values.astype(float)) / 4,
        running_speed,
    ])
    
    # Treat each trial as a single time step (sequence length=1 per trial)
    # Or group consecutive trials into sequences
    seq_len = 20  # group 20 consecutive trials
    n_trials = stim_inputs.shape[0]
    n_seqs = n_trials // seq_len
    
    X_seqs = stim_inputs[:n_seqs * seq_len].reshape(n_seqs, seq_len, -1)
    Y_seqs = target_dff[:, :n_seqs * seq_len].T.reshape(n_seqs, seq_len, -1)
    
    X_tensor = torch.FloatTensor(X_seqs).to(device)
    Y_tensor = torch.FloatTensor(Y_seqs).to(device)
    
    # Train/test split
    n_train = int(0.8 * n_seqs)
    X_train, X_test = X_tensor[:n_train], X_tensor[n_train:]
    Y_train, Y_test = Y_tensor[:n_train], Y_tensor[n_train:]
    
    # Initialize and train
    pred_model = PredictiveRNN(n_input=5, n_hidden=256, n_output=n_target).to(device)
    optimizer2 = optim.Adam(pred_model.parameters(), lr=1e-3)
    
    print(f"Training data-driven RNN: {n_train} sequences of length {seq_len}")
    print(f"Predicting {n_target} cells from 5 stimulus features")
    
    losses2 = []
    for epoch in range(200):
        pred_model.train()
        pred, hidden = pred_model(X_train)
        loss = nn.MSELoss()(pred, Y_train)
        
        optimizer2.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(pred_model.parameters(), 1.0)
        optimizer2.step()
        
        losses2.append(loss.item())
        if (epoch + 1) % 50 == 0:
            pred_model.eval()
            with torch.no_grad():
                pred_test, _ = pred_model(X_test)
                test_loss = nn.MSELoss()(pred_test, Y_test).item()
            print(f"  Epoch {epoch+1}: train_loss={losses2[-1]:.4f}, test_loss={test_loss:.4f}")
    
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(losses2, 'b-', linewidth=2)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss')
    ax.set_title('F2: Training Loss — Data-Driven RNN', fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ Skipping — PyTorch not available.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# F2.2  Compare RNN hidden representations to real population data
#       using CKA (Centered Kernel Alignment) and RSA
# ══════════════════════════════════════════════════════════════════════

if TORCH_AVAILABLE:
    from sklearn.decomposition import PCA
    
    pred_model.eval()
    
    # ── Get hidden states and predictions on test set ──
    with torch.no_grad():
        pred_test, hidden_test = pred_model(X_test)
    
    pred_np = pred_test.cpu().numpy().reshape(-1, n_target)  # (n_test_trials, n_cells)
    hidden_np = hidden_test.cpu().numpy().reshape(-1, 256)   # (n_test_trials, n_hidden)
    actual_np = Y_test.cpu().numpy().reshape(-1, n_target)   # (n_test_trials, n_cells)
    
    # Subsample time points for efficiency
    n_sub = min(500, actual_np.shape[0])
    idx = np.random.choice(actual_np.shape[0], n_sub, replace=False)
    
    cka_actual_hidden = linear_CKA(actual_np[idx], hidden_np[idx])
    cka_actual_pred = linear_CKA(actual_np[idx], pred_np[idx])
    
    print(f"CKA (real activity vs RNN hidden): {cka_actual_hidden:.4f}")
    print(f"CKA (real activity vs RNN predicted): {cka_actual_pred:.4f}")
    
    # ── PCA comparison: real vs RNN hidden ──
    pca_real = PCA(n_components=3).fit_transform(actual_np[idx])
    pca_hidden = PCA(n_components=3).fit_transform(hidden_np[idx])
    
    # Get stimulus labels for coloring
    test_stim = X_test.cpu().numpy().reshape(-1, 5)[idx]
    test_ori_label = np.round(np.rad2deg(np.arctan2(test_stim[:, 1], test_stim[:, 0])) % 360).astype(int)
    
    fig, axes = plt.subplots(1, 3, figsize=(20, 5))
    
    ax = axes[0]
    sc0 = ax.scatter(pca_real[:, 0], pca_real[:, 1], c=test_ori_label, cmap='hsv', s=10, alpha=0.5)
    ax.set_title('Real Population PCA', fontweight='bold')
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    plt.colorbar(sc0, ax=ax, label='Orientation (°)')
    
    ax = axes[1]
    sc1 = ax.scatter(pca_hidden[:, 0], pca_hidden[:, 1], c=test_ori_label, cmap='hsv', s=10, alpha=0.5)
    ax.set_title('RNN Hidden States PCA', fontweight='bold')
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    plt.colorbar(sc1, ax=ax, label='Orientation (°)')
    
    # Learned recurrent weights analysis
    ax = axes[2]
    rnn_weights = pred_model.rnn.weight_hh_l0.detach().cpu().numpy()
    im = ax.imshow(rnn_weights, aspect='auto', cmap='RdBu_r',
                   vmin=-np.percentile(np.abs(rnn_weights), 95),
                   vmax=np.percentile(np.abs(rnn_weights), 95))
    plt.colorbar(im, ax=ax)
    ax.set_title('F2: Learned Recurrent Weights', fontweight='bold')
    ax.set_xlabel('From unit')
    ax.set_ylabel('To unit')
    
    plt.suptitle(f'F2: Data-Driven RNN Analysis (CKA={cka_actual_hidden:.3f})', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # ── Cell-by-cell prediction quality ──
    per_cell_r = []
    for ci in range(n_target):
        r, _ = pearsonr(actual_np[:, ci], pred_np[:, ci])
        per_cell_r.append(r)
    per_cell_r = np.array(per_cell_r)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    ax = axes[0]
    ax.hist(per_cell_r, bins=30, color='steelblue', alpha=0.7, edgecolor='k')
    ax.axvline(np.median(per_cell_r), color='red', ls='--', label=f'median={np.median(per_cell_r):.3f}')
    ax.set_xlabel('Pearson r (predicted vs actual)')
    ax.set_ylabel('# Cells')
    ax.set_title('F2: Per-Cell Prediction Quality', fontweight='bold')
    ax.legend()
    
    # By subclass
    ax = axes[1]
    pred_df = pd.DataFrame({'r': per_cell_r, 'subclass': target_subclasses})
    for sc in present_subclasses:
        vals = pred_df.loc[pred_df['subclass'] == sc, 'r'].values
        if len(vals) >= 3:
            ax.scatter([SUBCLASS_SHORT[sc]] * len(vals), vals, alpha=0.5, s=20, c=SUBCLASS_COLORS[sc])
            ax.scatter(SUBCLASS_SHORT[sc], np.mean(vals), s=100, c=SUBCLASS_COLORS[sc],
                      edgecolors='k', zorder=5, marker='D')
    ax.set_xlabel('Subclass')
    ax.set_ylabel('Prediction r')
    ax.set_title('F2: Prediction Quality by Cell Type', fontweight='bold')
    ax.tick_params(axis='x', rotation=45)
    ax.axhline(0, color='k', ls='--', alpha=0.4)
    
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ Skipping — PyTorch not available.")